# Day 23: RAG, Embeddings and Vector DBs
**Grounded generation, embeddings, approximate nearest neighbour search, hybrid search and re-ranking**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai scikit-learn numpy pandas faiss-cpu rank_bm25

# ===== Option B: VS Code / local =====
# pip install openai scikit-learn numpy pandas faiss-cpu rank_bm25
# Create a .env file or set the variable in your terminal:  OPENAI_API_KEY=sk-...
# No key? Everything still runs in OFFLINE demo mode with a tiny built-in stand-in model.
print("Setup done")

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
"""
import json, os, re
import numpy as np

def _get_key():
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")                 # change to any current chat model you have access to
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None):
    if ONLINE:
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}])
        return r.choices[0].message.content
    return offline_llm(prompt, system)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, ONLINE

## 2. Documents and chunks

In [ ]:
%%writefile knowledge.md
# Sakura Logistics Employee Handbook (sample)

## 1. Working hours
Standard working hours are 9:30 to 18:30, Monday to Friday. Teams in the Tokyo office follow JST and teams in the Bengaluru office follow IST.
Flexible start times between 8:00 and 11:00 are allowed with manager approval.

## 2. Leave policy
Full-time employees receive 18 days of paid leave per year. Leave accrues monthly at 1.5 days.
Unused leave can be carried forward up to a maximum of 10 days. Carried forward leave expires on 31 March of the following year.
Sick leave is separate: employees receive 8 days of paid sick leave per year. A medical certificate is required for sick leave longer than 2 consecutive days.

## 3. Remote work
Employees may work remotely up to 2 days per week. Remote work must be logged in the HR portal by 10:00 on the same day.
Remote work from another country requires approval from HR and the security team at least 14 days in advance.

## 4. Expenses
Travel expenses must be submitted within 30 days with receipts. Meals during business travel are reimbursed up to 2,000 INR or 4,000 JPY per day.
Expenses above 50,000 INR require approval from a director before booking.

## 5. Security
Laptops must use full disk encryption. Customer data must never be copied to personal devices or uploaded to unapproved AI tools.
Security incidents must be reported to security@sakura-logistics.example within 1 hour of discovery.

## IT FAQ
Error VPN-417 means your VPN certificate has expired. Renew it from the IT self-service portal and restart the VPN client.
Password resets are done through the self-service portal. Passwords must be at least 14 characters long.
Approved AI tools are listed on the IT intranet page. Do not paste customer data into tools that are not on that list.
New laptops are issued within 5 working days of a request approved by your manager.

## Travel FAQ
Book flights through the corporate travel desk at least 10 days before travel. Economy class is standard for flights under 6 hours.
Hotel stays are reimbursed up to 6,000 INR per night in India and 15,000 JPY per night in Japan.
Visa support letters are issued by HR within 3 working days.

In [ ]:
import re, time, numpy as np, pandas as pd
text = open("knowledge.md").read()
chunks = []
for sec in re.split(r"\n(?=## )", text):
    lines = sec.strip().splitlines()
    if not lines or not lines[0].startswith("## "):
        continue
    title = lines[0][3:].strip()
    for s in re.split(r"(?<=[.!?])\s+", " ".join(lines[1:]).strip()):
        if s:
            chunks.append({"id": f"c{len(chunks) + 1}", "section": title, "text": s})
texts = [f"{c['section']}: {c['text']}" for c in chunks]
print(len(chunks), "chunks. Example:", chunks[0])

## 3. Embeddings and cosine similarity

In [ ]:
sents = ["How many leave days do I get?", "Employees receive 18 days of paid leave per year.",
         "What is the hotel limit in Tokyo?", "Hotel stays are reimbursed up to 15,000 JPY per night in Japan.",
         "How long must my password be?", "Passwords must be at least 14 characters long."]
V = embed(sents)
sim = pd.DataFrame(V @ V.T, index=[s[:22] for s in sents], columns=[f"s{i}" for i in range(6)]).round(2)
print(sim)
if not ONLINE:
    print("\nOffline embeddings only match shared words. With a real model, questions match their answers by meaning.")

## 4. A vector index with FAISS

In [ ]:
import faiss
vecs = embed(texts).astype("float32")
index = faiss.IndexFlatIP(vecs.shape[1])
index.add(vecs)

def vector_search(q, k=5):
    D, I = index.search(embed(q).astype("float32"), k)
    return [chunks[i]["id"] for i in I[0]]

for q in ["hotel limit in Japan", "VPN-417 error", "carry forward leave"]:
    print(f"{q:<22} -> {vector_search(q, 3)}")

## 5. Exact vs approximate search at scale

In [ ]:
rng = np.random.default_rng(23)
X = rng.normal(size=(20000, 128)).astype("float32"); X /= np.linalg.norm(X, axis=1, keepdims=True)
Q = rng.normal(size=(200, 128)).astype("float32"); Q /= np.linalg.norm(Q, axis=1, keepdims=True)

flat = faiss.IndexFlatIP(128); flat.add(X)
t = time.perf_counter(); _, I_flat = flat.search(Q, 10); t_flat = (time.perf_counter() - t) / len(Q) * 1000
print(f"Flat            {t_flat:.3f} ms/query  recall 1.00")

hnsw = faiss.IndexHNSWFlat(128, 32, faiss.METRIC_INNER_PRODUCT)
hnsw.hnsw.efConstruction = 80
hnsw.add(X)
for ef in [16, 64, 256]:                 # efSearch: how widely HNSW explores per query
    hnsw.hnsw.efSearch = ef
    t = time.perf_counter(); _, I_h = hnsw.search(Q, 10); t_h = (time.perf_counter() - t) / len(Q) * 1000
    recall = np.mean([len(set(a) & set(b)) / 10 for a, b in zip(I_flat, I_h)])
    print(f"HNSW efSearch={ef:<4} {t_h:.3f} ms/query  recall {recall:.2f}")
print("Random vectors are a hard case. Real embeddings cluster by topic, so ANN recall is usually higher at the same speed.")

## 6. Keyword search with BM25

In [ ]:
from rank_bm25 import BM25Okapi
tok = lambda s: re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", s.lower())
bm25 = BM25Okapi([tok(t) for t in texts])

def bm25_search(q, k=5):
    scores = bm25.get_scores(tok(q))
    return [chunks[i]["id"] for i in np.argsort(scores)[::-1][:k]]

for q in ["VPN-417", "14 characters", "visa letter"]:
    print(f"{q:<14} bm25 {bm25_search(q, 3)}   vector {vector_search(q, 3)}")

## 7. Hybrid search with reciprocal rank fusion, and measuring it

In [ ]:
def rrf(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, 1):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

def hybrid_search(q, k=5):
    return rrf([bm25_search(q, 20), vector_search(q, 20)])[:k]

by_text = {c["text"]: c["id"] for c in chunks}
find = lambda frag: next(c["id"] for c in chunks if frag in c["text"])
tests = [("How many paid leave days do full-time staff get?", find("18 days of paid leave")),
         ("When does carried over leave expire?", find("expires on 31 March")),
         ("What does error VPN-417 mean?", find("VPN-417")),
         ("How long must a password be?", find("14 characters")),
         ("Hotel limit per night in Japan?", find("15,000 JPY")),
         ("How soon must I book flights?", find("10 days before travel")),
         ("Deadline to submit travel expenses?", find("within 30 days")),
         ("Who issues visa support letters?", find("Visa support letters"))]

def hit_rate(search, k=3):
    return np.mean([gold in search(q, k) for q, gold in tests])

print(pd.Series({"bm25": hit_rate(bm25_search), "vector": hit_rate(vector_search), "hybrid": hit_rate(hybrid_search)}, name="hit@3").round(2))

## 8. Re-ranking

In [ ]:
def rerank(q, ids, keep=3):
    by_id = {c["id"]: c for c in chunks}
    scored = []
    for cid in ids:
        c = by_id[cid]
        if ONLINE:
            r = llm(f"Rate 0 to 10 how well this passage answers the question. Reply with one integer.\nQuestion: {q}\nPassage: {c['text']}", temperature=0)
            s = float(re.findall(r"\d+", r)[0]) / 10 if re.findall(r"\d+", r) else 0.0
        else:
            qw, cw = set(tok(q)), set(tok(c["section"] + " " + c["text"]))
            s = len(qw & cw) / max(1, len(qw))           # offline stand-in for a cross-encoder
        scored.append((cid, round(s, 2)))
    return sorted(scored, key=lambda x: x[1], reverse=True)[:keep]

q = "When does carried over leave expire?"
cands = hybrid_search(q, 8)
print("Candidates:", cands)
print("Re-ranked: ", rerank(q, cands))

## 9. Grounded answers with a refusal threshold

In [ ]:
RULES = "Answer using ONLY the context. Cite chunk ids like [c3]. If the context does not contain the answer, say \"I don't know\"."
by_id = {c["id"]: c for c in chunks}

def ask(q, threshold=0.3):
    top = rerank(q, hybrid_search(q, 8))
    if not top or top[0][1] < threshold:
        return "I don't know (no relevant passage found).", top
    context = "\n".join(f"[{cid}] {by_id[cid]['text']}" for cid, _ in top)
    return llm(f"Context:\n{context}\n\nQuestion: {q}", system=RULES, temperature=0), top

for q in ["What does error VPN-417 mean?", "Hotel limit per night in Japan?", "Can I bring my dog to the office?"]:
    a, top = ask(q)
    print(f"Q: {q}\nA: {a}\n   evidence: {top}\n")

### TODO 1: Add two more test questions (one unanswerable) and tune `threshold` so answerable questions are answered and the unanswerable one is refused.

In [ ]:
# TODO: write your code here

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)